In [0]:
# Create autoloader_input folder in Volume
dbutils.fs.mkdirs("/Volumes/dev/bronze/landing/autoloader_input/2010/12/01"); 
dbutils.fs.mkdirs("/Volumes/dev/bronze/landing/autoloader_input/2010/12/02")
dbutils.fs.mkdirs("/Volumes/dev/bronze/landing/autoloader_input/2010/12/03")
dbutils.fs.mkdirs("/Volumes/dev/bronze/landing/autoloader_input/2010/12/04")
dbutils.fs.mkdirs("/Volumes/dev/bronze/landing/autoloader_input/2010/12/05")
dbutils.fs.mkdirs("/Volumes/dev/bronze/landing/autoloader_input/2010/12/06")
dbutils.fs.mkdirs("/Volumes/dev/bronze/landing/autoloader_input/2010/12/07")

In [0]:
# Create checkpoint location in Volume
# Auto loaders uses checkpointing to manage incremental load from cloudpoint location
dbutils.fs.mkdirs("/Volumes/dev/bronze/landing/autoloader_checkpoint/autoloader");

In [0]:
# copy files to nested location

dbutils.fs.cp("/databricks-datasets/definitive-guide/data/retail-data/by-day/2010-12-01.csv", "/Volumes/dev/bronze/landing/autoloader_input/2010/12/01");
dbutils.fs.cp("/databricks-datasets/definitive-guide/data/retail-data/by-day/2010-12-02.csv", "/Volumes/dev/bronze/landing/autoloader_input/2010/12/02");
dbutils.fs.cp("/databricks-datasets/definitive-guide/data/retail-data/by-day/2010-12-03.csv", "/Volumes/dev/bronze/landing/autoloader_input/2010/12/03");

In [0]:
# Autoloader uses rocksDB to manage checkpointing
# Autoloader uses a location to store schema for schema evolution
df = (spark.readStream.format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("pathGlobFilter", "*.csv")
      .option("header","true")
      .option("cloudFiles.schemaHints","Quantity int, UnitPrice double")
      .option("cloudFiles.schemaLocation", "/Volumes/dev/bronze/landing/autoloader_checkpoint/autoloader/1/")
      .load("/Volumes/dev/bronze/landing/autoloader_input/*/"))



In [0]:
# write data to delta table -
# trigger(availableNow=True) --> this will help to run the data load in batch mode
# mergeSchema --> this will help to add new fields to the table

from pyspark.sql.functions import col
(df
 .withColumn("__file",col("_metadata.file_name"))
 .writeStream
 .option("checkpointLocation", "/Volumes/dev/bronze/landing/autoloader_checkpoint/autoloader/1/")
 .option("mergeSchema","true")
 .outputMode("append")
 .trigger(availableNow=True)
.toTable("dev.bronze.invoice_al_1"))

In [0]:
%sql
select * from dev.bronze.invoice_al_1;

In [0]:
dbutils.fs.cp("/databricks-datasets/definitive-guide/data/retail-data/by-day/2010-12-05.csv", "/Volumes/dev/bronze/landing/autoloader_input/2010/12/05");

In [0]:
#### Rerun the read df and write command and check if it's creating duplicate records

In [0]:
%sql
select __file, count(1)
from dev.bronze.invoice_al_1
group by __file

### RESCUE OPTION

In [0]:
# Rescue option - new or extra fields value will get pushed tp _rescued_data column
df2 = (spark.readStream.format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("pathGlobFilter", "*.csv")
      .option("header","true")
      .option("cloudFiles.schemaHints","Quantity int, UnitPrice double")
      .option("cloudFiles.schemaLocation", "/Volumes/dev/bronze/landing/autoloader_checkpoint/autoloader/2/")
      .option("cloudFiles.schemaEvolutionMode", "rescue")
      .load("/Volumes/dev/bronze/landing/autoloader_input/*/"))

In [0]:
from pyspark.sql.functions import col
(df2
 .withColumn("__file",col("_metadata.file_name"))
 .writeStream
 .option("checkpointLocation", "/Volumes/dev/bronze/landing/autoloader_checkpoint/autoloader/2/")
 .option("mergeSchema","true")
 .outputMode("append")
 .trigger(availableNow=True)
.toTable("dev.bronze.invoice_al_2"))

### evolution mode - None (Ignores new fields means no schema evolution and new fields are ignored)

In [0]:
df3 = (spark.readStream.format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("pathGlobFilter", "*.csv")
      .option("header","true")
      .option("cloudFiles.schemaHints","Quantity int, UnitPrice double")
      .option("cloudFiles.schemaLocation", "/Volumes/dev/bronze/landing/autoloader_checkpoint/autoloader/3/")
      .option("cloudFiles.schemaEvolutionMode", "none")
      .load("/Volumes/dev/bronze/landing/autoloader_input/*/"))

In [0]:
from pyspark.sql.functions import col
(df3
 .withColumn("__file",col("_metadata.file_name"))
 .writeStream
 .option("checkpointLocation", "/Volumes/dev/bronze/landing/autoloader_checkpoint/autoloader/3/")
 .option("mergeSchema","true")
 .outputMode("append")
 .trigger(availableNow=True)
.toTable("dev.bronze.invoice_al_3"))